# Static Maps and Charts

Every map in this course so far has been **interactive** – `.explore()` in the early modules, Folium in the [previous section](map_1.ipynb). Those are the right tool for exploring data and for publishing on the web, and they are useless the moment you need a figure in a report, a thesis or a paper.

For that you need a **static** map: an image file, at a fixed size, with a legend that prints. That is what `matplotlib` is for, and GeoPandas plots straight into it.

In this section we build one from the ground up, put a chart beside it, and make the two agree on their colours.


## 0. Importing Libraries


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily as ctx

- [**matplotlib**](https://matplotlib.org/) (`matplotlib.pyplot`) – the plotting library the whole scientific Python stack draws with. GeoPandas' `.plot()` is a thin layer on top of it, so every matplotlib setting is available to a map.

- [**contextily**](https://contextily.readthedocs.io/) (`contextily`) – fetches basemap tiles and draws them under a static map, the way the tile layer sits under a Folium map.


## 1. A First Map

Every `GeoDataFrame` has a `.plot()` method. It is the static counterpart of `.explore()`, and the quickest way to see what you are holding.

### 1.1. Loading the Data

We use the district boundaries of Vienna – the same file as in the third module. _Every dataset and its source is listed on the [Course Modules](../module_0/syllabus.md) page._


In [ ]:
districts = gpd.read_file("../../data/vienna/vienna_admin.gpkg", layer="district")

districts.head()

### 1.2. The Default Call

Called with no arguments at all, `.plot()` picks a colour and a size for you. It is not a finished map, but it answers "did the file load and is it where I think it is" in one line.


In [ ]:
districts.plot()
plt.show()

### 1.3. Figure and Axes

To go further you need the two objects matplotlib is built on:

- **Figure** (`fig`) – the sheet of paper. The whole image, including its margins.
- **Axes** (`ax`) – the frame drawn on it. One sheet can carry several frames, which is how we put a map and a chart side by side further down.

`plt.subplots()` creates both and hands them back as a pair. Passing `ax=ax` into `.plot()` is what tells GeoPandas **which frame to draw into** – without it, every call starts a new figure of its own.

| Parameter   | What it sets                  | Example     |
| :---------- | :---------------------------- | :---------- |
| `color`     | fill colour                   | `"#d0e8f1"` |
| `edgecolor` | boundary colour               | `"gray"`    |
| `linewidth` | boundary width                | `0.8`       |
| `alpha`     | opacity, 0 to 1               | `0.9`       |
| `figsize`   | figure size in inches, (w, h) | `(8, 7)`    |
| `ax`        | the frame to draw into        | `ax`        |


In [ ]:
# one sheet, 8 by 7 inches, carrying a single frame
fig, ax = plt.subplots(figsize=(8, 7))

districts.plot(
    ax=ax,
    color="#d0e8f1",
    edgecolor="#4a90a4",
    linewidth=0.8,
    alpha=0.9,
)

ax.set_title("The 23 districts of Vienna", fontsize=14, pad=12)
ax.axis("off")        # coordinate ticks mean nothing on a map - hide them

plt.tight_layout()    # trims the slack around the frame
plt.show()

### 1.4. Saving the Map

`fig.savefig()` writes the figure to a file, and the extension decides the format: `.png` for a document, `.svg` or `.pdf` when it has to stay sharp at any size.

| Parameter             | What it sets                                 |
| :-------------------- | :------------------------------------------- |
| `dpi`                 | resolution – 150 for a screen, 300 for print |
| `bbox_inches="tight"` | trims the white margin around the figure     |

One catch worth knowing before it costs you an afternoon: **`savefig()` must come before `plt.show()`**. Showing a figure clears it, and saving afterwards writes a blank image.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
districts.plot(ax=ax, color="#d0e8f1", edgecolor="#4a90a4", linewidth=0.8)
ax.set_title("The 23 districts of Vienna", fontsize=14)
ax.axis("off")
plt.tight_layout()

# save first ...
fig.savefig("vienna_districts.png", dpi=150, bbox_inches="tight")

# ... then show
plt.show()

## 2. Adding Points

### 2.1. Preparing the Data

We add the well-known places to visit from `vienna_top_locations.csv`. As in the first module, the file follows continental European conventions – semicolons between values, a comma as the decimal mark – so both have to be declared.


In [ ]:
locations_df = pd.read_csv(
    "../../data/vienna/vienna_top_locations.csv",
    sep=";",
    decimal=",",
).dropna(subset=["geo_longitude", "geo_latitude"])

locations = gpd.GeoDataFrame(
    locations_df,
    geometry=gpd.points_from_xy(
        locations_df["geo_longitude"], locations_df["geo_latitude"]
    ),
    crs="EPSG:4326",
)

print(f"{len(locations)} locations")
locations[["title", "category", "geometry"]].head()

### 2.2. Two Layers in One Frame

To draw several layers on one map, pass **the same `ax`** to each `.plot()` call. That is the whole mechanism: `ax=ax` means "into the frame that already exists" rather than "start a new figure".

Order matters – each layer is drawn on top of the ones before it, so the background goes first.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

# layer 1: the districts, as a backdrop
districts.plot(ax=ax, color="#f0f0f0", edgecolor="gray", linewidth=0.5)

# layer 2: the locations, into the same frame
locations.plot(ax=ax, color="#c1443c", markersize=20, marker="o", alpha=0.7)

ax.set_title("Top locations across Vienna", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

That map has a problem, and it is worth naming rather than styling away: **almost every point lands in the middle**. Vienna's notable places cluster in the historic centre, so at the scale of the whole city they collapse into a blob while three quarters of the sheet stays empty.

The fix is not a smaller marker. It is to map the area the data is actually about. The **inner districts** – 1 to 9, inside the Gürtel – hold 118 of the 135 locations, and at that scale the points separate.


In [ ]:
inner = districts[districts["BEZNR"] <= 9]

# keep the locations that fall inside them - a spatial filter, as in the third module
inner_locations = locations[locations.within(inner.geometry.union_all())]

print(f"{len(inner_locations)} of {len(locations)} locations are in districts 1-9")

### 2.3. Grouping the Categories

The `category` column carries eleven values, which is more than a legend can carry and more than the eye can separate. So we group them into five.

Note the two spellings of _museum_ in the mapping below. That is not a typo of ours: the published file contains both `museum` and `musuem`, exactly the kind of thing `value_counts()` was for in [Exploring a Dataset](../module_1/spData_3.ipynb). A dictionary that silently dropped the misspelling would quietly lose a feature.


In [ ]:
category_groups = {
    "museum": "Culture",
    "musuem": "Culture",          # the misspelling in the source file
    "musicstage": "Culture",
    "sightseeing": "Culture",
    "cafes": "Food and drink",
    "gastronomy": "Food and drink",
    "restaurants": "Food and drink",
    "shopping": "Shopping",
    "nightlife": "Nightlife",
    "accommodations": "Other",
    "servicepoints": "Other",
}

inner_locations = inner_locations.copy()
inner_locations["group"] = inner_locations["category"].map(category_groups)

# nothing should fall through the mapping
print("unmapped:", inner_locations["group"].isna().sum())
inner_locations["group"].value_counts()

### 2.4. Colouring by Category

`column=` colours features by the values of a field. For text values add `categorical=True`, and `legend=True` builds the legend from the unique values.

| Parameter          | What it sets                                |
| :----------------- | :------------------------------------------ |
| `column`           | the field to colour by                      |
| `categorical=True` | treat the values as categories, not numbers |
| `legend=True`      | draw a legend                               |
| `cmap`             | the colour scheme                           |
| `legend_kwds`      | a dictionary of legend settings             |

#### Choosing a colour scheme

Matplotlib's colormaps come in three kinds – the split is Brewer's, and it is the one every cartographic guide uses (Harrower & Brewer, 2003). Picking the wrong kind misleads the reader before they have read a single label:

| Kind            | Use it for                                | Examples                     |
| :-------------- | :---------------------------------------- | :--------------------------- |
| **Qualitative** | categories with no order                  | `Set2`, `tab10`, `Paired`    |
| **Sequential**  | numbers running low to high               | `Blues`, `YlOrRd`, `viridis` |
| **Diverging**   | values either side of a meaningful middle | `RdBu`, `coolwarm`, `PiYG`   |

Our groups have no order – _Culture_ is not more than _Shopping_ – so the scheme must be **qualitative**. A sequential scheme here would imply a ranking that does not exist.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

inner.plot(ax=ax, color="#f5f5f5", edgecolor="gray", linewidth=0.5)
inner_locations.plot(
    ax=ax,
    column="group",
    categorical=True,
    legend=True,
    cmap="Set2",
    markersize=40,
    alpha=0.9,
    legend_kwds={
        "loc": "upper left",
        "title": "Type of place",
        "framealpha": 0.85,
    },
)

ax.set_title("Top locations of central Vienna, by type", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Charts

A map answers **where**. It is poor at **how many**, because the eye cannot count scattered dots and cannot judge proportions from them at all. That is what a chart is for, and the two together say more than either alone.

### 3.1. Bar Chart

`ax.bar()` draws vertical bars, `ax.barh()` horizontal ones. Prefer horizontal when the category names are long: they stay readable instead of overlapping under the axis.


In [ ]:
group_counts = inner_locations["group"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))

ax.barh(group_counts.index, group_counts.values, color="#4a90a4")

ax.set_xlabel("Number of locations")
ax.set_title("Top locations by type")
ax.spines[["top", "right"]].set_visible(False)   # drop the unneeded frame

plt.tight_layout()
plt.show()

### 3.2. Pie Chart

A pie chart shows parts of a whole. It works when there are few slices – three to five – and fails past that, because the eye compares angles badly.

| Parameter    | What it sets                                 |
| :----------- | :------------------------------------------- |
| `autopct`    | the percentage label on each slice           |
| `startangle` | where the first slice begins (90 is the top) |
| `colors`     | one colour per slice                         |
| `wedgeprops` | slice styling, such as the dividing line     |


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.pie(
    group_counts.values,
    labels=group_counts.index,
    autopct="%1.0f%%",
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)

ax.set_title("Share of each type")
plt.tight_layout()
plt.show()

## 4. A Map and a Chart Together

`plt.subplots(nrows, ncols)` builds a grid of frames on one figure, which is how a map and its chart end up in a single image.

```
fig
+---------------------------------+
|  ax_map          |  ax_chart    |
|  (the map)       |  (the chart) |
+---------------------------------+
```

`plt.subplots(1, 2)` means one row, two columns, and returns the two frames together: `fig, (ax_map, ax_chart) = ...`. Each is then used exactly as the single `ax` was above.

### 4.1. The Naive Version

Built the obvious way, and with a fault worth seeing before we fix it.


In [ ]:
fig, (ax_map, ax_chart) = plt.subplots(1, 2, figsize=(14, 6))

# left frame: the map, coloured from Set2
inner.plot(ax=ax_map, color="#f0f0f0", edgecolor="gray", linewidth=0.4)
inner_locations.plot(ax=ax_map, column="group", categorical=True,
                     cmap="Set2", markersize=35, alpha=0.9)
ax_map.set_title("Locations by type", fontsize=12)
ax_map.axis("off")

# right frame: the chart, coloured from Set2 as well
ax_chart.barh(group_counts.index, group_counts.values,
              color=plt.cm.Set2.colors[:len(group_counts)])
ax_chart.set_title("How many of each", fontsize=12)
ax_chart.set_xlabel("Number of locations")
ax_chart.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

### 4.2. Making the Colours Agree

Both halves were given the **same palette**, and they still disagree. _Culture_ and _Food and drink_ happen to match; _Shopping_, _Nightlife_ and _Other_ are one colour among the dots and a different one on their bars.

The reason is that each half took the palette **in its own order**. GeoPandas sorts the categories alphabetically:

```
Culture, Food and drink, Nightlife, Other, Shopping
```

while `value_counts()` returns them largest first:

```
Culture, Food and drink, Shopping, Nightlife, Other
```

The first two coincide, so the figure looks right at a glance. From the third position on, the same colour means a different category on either side.

This is the kind of fault that survives review. Nothing errors, both halves look deliberate, and a reader who takes a colour from the chart and hunts for it on the map is sent to the wrong places entirely.

The fix is to stop letting either half choose. We decide once, in a dictionary, and both read from it.


In [ ]:
group_colors = {
    "Culture": "#4a90d9",
    "Food and drink": "#e8714a",
    "Shopping": "#5cb98c",
    "Nightlife": "#8856a7",
    "Other": "#9e9e9e",
}

The dictionary is applied in two places.

**On the map**, `.map()` turns the category column into a column of colours, one per feature, which goes straight into `color=`:

```python
inner_locations["group"].map(group_colors)
```

`.map()` is a pandas method that walks a column and replaces each value according to a rule – here, a dictionary. `"Culture"` becomes `"#4a90d9"` for every row that holds it.

**On the chart**, we build the list of colours in the order the bars are drawn:

```python
[group_colors[name] for name in group_counts.index]
```

Both now come from the same source, so they cannot drift apart – and if you change a colour later, you change it once.


In [ ]:
fig, (ax_map, ax_chart) = plt.subplots(1, 2, figsize=(14, 6))

# the map takes one colour per feature
inner.plot(ax=ax_map, color="#f5f5f5", edgecolor="gray", linewidth=0.4)
inner_locations.plot(
    ax=ax_map,
    color=inner_locations["group"].map(group_colors),
    markersize=40,
    alpha=0.9,
    edgecolor="white",
    linewidth=0.4,
)
ax_map.set_title("Locations by type", fontsize=12)
ax_map.axis("off")

# the chart takes the colours in bar order, from the same dictionary
bar_colors = [group_colors[name] for name in group_counts.index]
ax_chart.barh(group_counts.index, group_counts.values, color=bar_colors)
ax_chart.set_title("How many of each", fontsize=12)
ax_chart.set_xlabel("Number of locations")
ax_chart.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## 5. A Basemap Underneath

A static map has no basemap of its own, so a reader who does not know the city has nothing to place the points against. **contextily** fetches tiles and draws them under the data – the static equivalent of the tile layer in a Folium map.

One requirement: tile services publish in **Web Mercator (EPSG:3857)**, the projection of the [second module](../module_2/projections_1.ipynb). The data has to be reprojected to match, or it lands somewhere off the coast of Africa.

**Which tiles.** Not every provider is usable from a script, and the two you would reach for first are the two that fail:

- **OpenStreetMap's own tiles** return `403 Access blocked`. Their servers are volunteer-run and their usage policy excludes bulk automated fetching.
- **CartoDB Positron**, the basemap we use throughout the Folium sections, now stamps `API KEY REQUIRED` across tiles fetched without a key.

`Esri.WorldGrayCanvas` is free, needs no key, and is the light, quiet backdrop we want anyway. Note that contextily writes the tile attribution into the corner by itself – which, as [section 3.6](map_1.ipynb) put it, covers the basemap but never the data you draw on top.


In [ ]:
# tiles are published in Web Mercator, so the data has to meet them there
inner_3857 = inner.to_crs(epsg=3857)
locations_3857 = inner_locations.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(9, 8))

inner_3857.plot(ax=ax, color="none", edgecolor="#666666", linewidth=0.6, alpha=0.7)
locations_3857.plot(
    ax=ax,
    color=locations_3857["group"].map(group_colors),
    markersize=40,
    alpha=0.9,
    edgecolor="white",
    linewidth=0.4,
)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas)

ax.set_title("Top locations of central Vienna", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Summary

In this section we built static maps – the kind that goes into a document rather than a browser.

We learned:

- how `.plot()` draws a `GeoDataFrame`, and how **Figure and Axes** let you control what it draws into;
- how to stack layers by passing the same `ax`, and how to save the result with `savefig()`;
- how to colour features by category, and why the colour scheme has to be **qualitative** when the categories have no order;
- how to put a map and a chart on one figure, and why their colours must come from **one dictionary** rather than from two independent defaults;
- how to draw a basemap underneath with `contextily`, and which tile providers still work without a key.

One thing was decided rather than drawn: the first point map covered the whole city and was unreadable, so we mapped the inner districts instead. Choosing the extent is part of making the map, not a preliminary to it.

In the [next section](map_4.ipynb) we colour polygons by a **number** rather than a category – which raises the question this one did not have to answer: where do the class boundaries go?


## References

Harrower, M., & Brewer, C. A. (2003). ColorBrewer.org: An online tool for selecting colour schemes for maps. *The Cartographic Journal*, 40(1), 27–37. https://doi.org/10.1179/000870403235002042
